# Description
Reconstruction of Italgas Leak investigation from G2G

In [1]:
from locallib.picarrodb import *
from locallib.box import *
from locallib.query import *
from locallib.pandas import *

import sqlite3
import os
import pandas as pd
from inc import *

/home/sandbox/venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


# Connect to the Italgas databaes

In [2]:
DATABASE_PATH = os.path.join(os.getcwd(), "database/" "italgas_g2g_anders.db")
# Create a connection to the Italgas database using sqlite3
conn = sqlite3.connect(DATABASE_PATH)
print(DATABASE_PATH)

/home/sandbox/personal-repos/DA-3590/database/italgas_g2g_anders.db


In [3]:
query = 'SELECT * FROM LEAKS'
leaks_g2g = pd.read_sql_query(query, conn)
print(len(leaks_g2g))

239061


In [4]:
leaks_g2g['lisa'] = leaks_g2g['lisa'].str.replace('-LISA', '-L-')

In [5]:
# Set LeakFound and fill some missing values in leaks_g2g
leaks_g2g['LeakFound'] = ''
leaks_g2g.loc[leaks_g2g['codiceDispersione'].isin(['A1', 'A2', 'B', 'C', 'PRELOCALIZZATA']), 'LeakFound'] = 'Found_Gas_Leak'
leaks_g2g.loc[leaks_g2g['codiceDispersione'] == '', ['LeakFound', 'codiceDispersione']] = ['Not_Investigated', 'No Grade']
leaks_g2g.loc[leaks_g2g['aereoInterrato'] == '', 'aereoInterrato'] = 'NC'
leaks_g2g.loc[leaks_g2g['intervento'].isin([
    'INTERESSA IMPIANTI ALTRI SERVIZI-RP',
    'INTERESSA IMPIANTO ALTRO DISTRIBUTORE-RP'
]), 'LeakFound'] = 'Found_Other_Source'
leaks_g2g.loc[leaks_g2g['intervento'].isin([
    'COMUNE INDIRIZZO ERRATO-RP',
    'ANOMALIA NON RISCONTRATA-RP'
]), 'LeakFound'] = 'No_Gas_Found'

In [6]:
reportsQ = get_reports('Italgas',years = [2025,2026], table_name = '#TempReports')
emissionQ = emission_sources_table_query_given_report_id( report_table = '#TempReports', table_name = '#TempEmissionSources')
boxQ = query_box_table(report_table = '#TempReports', table_name = '#TempBoxes')
emissionQ.set_child(boxQ)
reportsQ.set_child(emissionQ)
data = reportsQ.execute(EU2_Conn, table_return = ['#TempReports','#TempEmissionSources','#TempBoxes'])

In [7]:
#data['#TempEmissionSources'] = data['#TempEmissionSources'].drop(columns = ['UniqueIdentifier'])
print(len(data['#TempEmissionSources']))

281984


In [8]:
G2G_COLS = leaks_g2g.columns.tolist()

In [9]:
data['#TempBoxes'] = data['#TempBoxes'].drop(columns = ['ReportId'])
data['#TempEmissionSources'] = data['#TempEmissionSources'].drop(columns = ['UniqueIdentifier'])

In [10]:
w_emission = pd.merge(data['#TempReports'], data['#TempEmissionSources'], left_on = 'ReportId', right_on = 'ReportId', how = 'left')
w_emission.emissionrates.convert()
w_emission.timezone.convert_utc_column_to_local('ReportDate', 'CommonTimeZone', 'ReportDateLocal')
print(len(w_emission))
w_box = pd.merge(w_emission, data['#TempBoxes'], left_on = 'EmissionSourceId', right_on = 'EmissionSourceId', how = 'left')
print(len(w_box))
full_df = pd.merge(w_box, leaks_g2g, left_on = 'UniqueIdentifier', right_on = 'lisa', how = 'left')
print(len(full_df))

282039
282040
287678


In [11]:
full_df.sort_values(by=['ReportId','RepresentativeBinLabel'],inplace=True,ascending=False)
full_df['UniqueBoundary'] = (~full_df.duplicated(subset='ReportId', keep='first')).astype(int)

# UniqueLISA: mark unique EmissionSourceId (first occurrence only) = 1
full_df['UniqueLISA'] = (~full_df.duplicated(subset='EmissionSourceId', keep='first')).astype(int)

# DuplicatedPeakInDifferentReports
dup_mask = ~full_df[["ReportId", "RepresentativePeakId"]].duplicated()

dup_mask &= full_df.drop_duplicates(subset=["ReportId", "RepresentativePeakId"]) \
            .duplicated(subset="RepresentativePeakId", keep=False) \
            .fillna(True)

full_df['DuplicatedPeakInDifferentReports'] = dup_mask.astype(int)

In [12]:
columns_list = [
    'CustomerName', 'ReportName', 'ReportTitle', 'ReportDate', 'ReportDateLocal', 'EmissionSourceId', 'LisaWkt4326', 
    'LisaNumber', 'UniqueIdentifier', 'ReportBuildNumber', 'ReportAssetLengthKm', 'ReportPercentCoverageAssets',
    'AssetCoveredLengthKm', 'CH4', 'ClassificationConfidence', 'Disposition', 'DetectionProbability', 'EmissionRate',
    'EmissionRateGramsPerHour', 'EmissionRateLPM', 'EmissionRateAMean', 'EmissionRateAStd', 'EmissionRateGMean',
    'EmissionRateGStd', 'EmissionRateLowerBound', 'EmissionRateUpperBound', 'EthaneRatio', 'EthaneRatioUncertainty',
    'GeocodeAddress', 'GpsLatitude', 'GpsLongitude', 'IsFiltered', 'MaxAmplitude', 'MaxCarSpeed', 'MaxWindSpeed',
    'MinWindSpeed', 'NumberOfPasses', 'NumberOfPeaks', 'PeakNumber', 'PriorityScore', 'RankingGroup', 'ReportId',
    'RepresentativePeakId', 'RepresentativeEmissionRate', 'RepresentativeEmissionRateGramsPerHour',
    'RepresentativeEmissionRateLPM', 'RepresentativeBinLabel', 'RepresentativePeakEpochTime', 'Labels',
    'NumberOfLabels'
] + G2G_COLS + ['UniqueBoundary', 'UniqueLISA', 'DuplicatedPeakInDifferentReports']


In [ ]:
full_df[columns_list].to_excel('Itakgas_Emissions_G2G.xlsx', index=False)

In [ ]:
box_obj = BoxFile(local_path = 'Itakgas_Emissions_G2G.xlsx', box_folder_id = 380811371684)
box_obj.upload()
box_obj.delete()